/Volumes/workspace/data/volume1/data1/

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
Orders =spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load("/Volumes/workspace/data/volume1/data1/orders_csv.csv")


In [0]:
Users =spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load("/Volumes/workspace/data/volume1/data1/customers_csv.csv")

In [0]:
New_orders = spark.read.format("json") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load("/Volumes/workspace/data/volume1/data1/new_orders.json")


spark.sql("query")

1. What is the total number of orders placed by each user?
spark.sql("select customer_id, count(*) as total_orders from orders group by customer_id").show()
2. What is the total revenue generated by each user?
spark.sql("select customer_id, sum(price) as total_revenue from orders group by customer_id").show()
3. What is the total revenue generated by each user in the last month?
spark.sql("select customer_id, sum(price) as total_revenue from orders where order_time >= date_sub(current_date(),30) group by customer_id").show()
4. What is the total revenue generated by each user in the last month, grouped by cuisine?
spark.sql("select cuisine, customer_id, sum(price) as total_revenue from orders join restaurants on orders.restaurant_id = restaurants.restaurant_id where order_time >= date_sub(current_date(),30) group by cuisine, customer_id").show()


In [0]:
#convert dataframe into temporary schema table
Orders.createOrReplaceTempView("Orders")
Users.createOrReplaceTempView("Users")
New_orders.createOrReplaceTempView("New_orders")


In [0]:
result = spark.sql("""
                   select * from Orders where amount>300
                   """)
result.show()

+--------+-----------+----------+------+--------+---------+----------+
|order_id|customer_id|product_id|amount|quantity|   status|order_date|
+--------+-----------+----------+------+--------+---------+----------+
|       1|        101|      1001|   500|       2|Completed|2024-01-01|
|       3|        103|      1003|   700|       3|Completed|2024-01-03|
|       5|        104|      1001|  1000|       4|Completed|2024-01-05|
+--------+-----------+----------+------+--------+---------+----------+



In [0]:
Orders.show()

+--------+-----------+----------+------+--------+---------+----------+
|order_id|customer_id|product_id|amount|quantity|   status|order_date|
+--------+-----------+----------+------+--------+---------+----------+
|       1|        101|      1001|   500|       2|Completed|2024-01-01|
|       2|        102|      1002|   300|       1|  Pending|2024-01-02|
|       3|        103|      1003|   700|       3|Completed|2024-01-03|
|       4|        101|      1002|   200|       1|Cancelled|2024-01-04|
|       5|        104|      1001|  1000|       4|Completed|2024-01-05|
|       6|        105|      1004|   150|       1|  Pending|2024-01-06|
+--------+-----------+----------+------+--------+---------+----------+



In [0]:
Users.show()

+-----------+------+-------+-----------+
|customer_id|  name|country|signup_date|
+-----------+------+-------+-----------+
|        101| Kusum|  India| 2023-12-01|
|        102|Nikesh|  India| 2023-12-05|
|        103|  Nick|    USA| 2023-12-10|
|        104|Sujana|     UK| 2023-12-15|
|        105| Nisha| Canada| 2023-12-20|
+-----------+------+-------+-----------+



In [0]:
# join + aggregration

revenue_df = spark.sql("""
                       SELECT 
                       u.country,
                       count(o.order_id) as total_orders,
                       sum(o.amount) as total_revenue
                       FROM Orders O
                       JOIN Users u
                       ON o.customer_id = u.customer_id
                       GROUP BY u.country
                       ORDER BY total_revenue DESC
                       """)

In [0]:
revenue_df.show()

+-------+------------+-------------+
|country|total_orders|total_revenue|
+-------+------------+-------------+
|  India|           3|         1000|
|     UK|           1|         1000|
|    USA|           1|          700|
| Canada|           1|          150|
+-------+------------+-------------+



In [0]:
min_amount = 300

filtered_df= spark.sql(f"""
                       SELECT * 
                       FROM Orders
                       WHERE amount > {min_amount}
                       """)

In [0]:
filtered_df.show()

+--------+-----------+----------+------+--------+---------+----------+
|order_id|customer_id|product_id|amount|quantity|   status|order_date|
+--------+-----------+----------+------+--------+---------+----------+
|       1|        101|      1001|   500|       2|Completed|2024-01-01|
|       3|        103|      1003|   700|       3|Completed|2024-01-03|
|       5|        104|      1001|  1000|       4|Completed|2024-01-05|
+--------+-----------+----------+------+--------+---------+----------+




write spark.sql queries with help of window function

In [0]:
New_orders.printSchema()

root
 |-- amount: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- status: string (nullable = true)



In [0]:
New_orders.show()

+------+-----------+--------+----------+--------+---------+
|amount|customer_id|order_id|product_id|quantity|   status|
+------+-----------+--------+----------+--------+---------+
|   750|        103|       3|      1003|       3|Completed|
|   400|        106|       7|      1005|       2|Completed|
+------+-----------+--------+----------+--------+---------+



In [0]:
#update and insert

#find new records

new_records =spark.sql("""
                       SELECT n.*
                       FROM New_orders n
                       LEFT JOIN Orders o
                       ON n.order_id = o.order_id
                       WHERE o.order_id IS NULL

                       """)

In [0]:
new_records.show()

+------+-----------+--------+----------+--------+---------+
|amount|customer_id|order_id|product_id|quantity|   status|
+------+-----------+--------+----------+--------+---------+
|   400|        106|       7|      1005|       2|Completed|
+------+-----------+--------+----------+--------+---------+



In [0]:
#find updated records
#CDC(Change Data Capture)
updated_records = spark.sql("""
                            SELECT n.*
                            FROM New_orders n
                            JOIN Orders O
                            ON n.order_id = O.order_id
                            """)

In [0]:

updated_records.show()

+------+-----------+--------+----------+--------+---------+
|amount|customer_id|order_id|product_id|quantity|   status|
+------+-----------+--------+----------+--------+---------+
|   750|        103|       3|      1003|       3|Completed|
+------+-----------+--------+----------+--------+---------+



In [0]:
#Delta tables